# 🧬 CryoAtom2: From cryo-EM density map to atomic structure

<img src="https://raw.githubusercontent.com/YangLab-SDU/CryoAtom/master/imgs/framework.png"
     width="600"
     align="right"
     style="margin-left: 20px; border: 2px solid #ccc; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">

CryoAtom2 is a software tool that automatically constructs all-atom models of proteins, nucleic acids, or their complexes from cryo-EM density maps and sequence information.

Paper available in [**Nature Structural & Molecular Biology**](https://www.nature.com/articles/s41594-025-01713-3). If you encounter any bugs, please report the issue to [**GitHub Issues**](https://github.com/YangLab-SDU/CryoAtom/issues).

[**YangLab-SDU**](https://yanglab.qd.sdu.edu.cn) focuses on computational biology and structural bioinformatics. Follow us on [**GitHub**](https://github.com/YangLab-SDU).

<br clear="all">

---

> ⚠️ **Important**: Please run the cells in order (`Step1` → `Step2` → `Step3`→ `Step4`). Do not use `Runtime` → `Run all`, as some steps require the previous one to finish first.
>
> ℹ️ Please scroll to the **very bottom of this page** to view the usage instructions and mode descriptions before proceeding.

In [ ]:
#@title # Step 1: Install dependencies (~6mins) { display-mode: "form" }
import os
import sys
import json
import subprocess
import importlib.util
import importlib
import re
import torch
from google.colab import drive, output
from IPython.display import HTML, display

# --- UI Component  ---
progress_html = """
<div id="setup-portal" style="padding: 20px; background: #ffffff; border: 1px solid #e0e0e0; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05); max-width: 1000px;">
    <h3 style="color: #1a73e8; margin-top: 0; border-bottom: 2px solid #e8f0fe; padding-bottom: 10px;">CryoAtom Environment Setup</h3>
    <div id="setup_progress_container" style="margin-top: 15px; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #1a73e8;">
        <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
            <span id="setup_status_msg" style="font-size: 13px; color: #1a73e8; font-weight: bold;">Initializing...</span>
            <span id="setup_progress_percent" style="font-size: 13px; color: #666;">0%</span>
        </div>
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; overflow: hidden;">
            <div id="setup_progress_bar" style="width: 0%; height: 100%; background-color: #2ecc71; transition: width 0.3s;"></div>
        </div>
    </div>
</div>
<script>
    window.updateSetupProgress = function(percent, message, color='#2ecc71') {
        const bar = document.getElementById('setup_progress_bar');
        const msg = document.getElementById('setup_status_msg');
        const pct = document.getElementById('setup_progress_percent');
        if(bar){ bar.style.width = percent + '%'; bar.style.backgroundColor = color; }
        if(msg){ msg.innerText = message; }
        if(pct){ pct.innerText = percent + '%'; }
    }
</script>
"""
display(HTML(progress_html))

def set_progress(p, m, c='#2ecc71'):
    try: output.eval_js(f"updateSetupProgress({p}, {json.dumps(m)}, {json.dumps(c)})")
    except: pass

# --- Configuration ---
target_a = "/content/CryoAtom/CryoAtom2/checkpoint"
target_b = "/root/.cache/torch/hub/checkpoints"
group_a = ["RUNet.pth", "CryoNet.pth", "CryoNet_no_seq.pth"]
group_b = ["esm2_t33_650M_UR50D-contact-regression.pt", "esm2_t33_650M_UR50D.pt", "RNA-FM_pretrained.pth"]

pyhmmer_ver_str = "pyhmmer>=0.10.1" if sys.version_info >= (3, 12) else "pyhmmer==0.7.1"

deps = [
    ("Bio", "biopython==1.81"), ("mrcfile", "mrcfile==1.4.3"), ("einops", "einops==0.6.0"),
    ("esm", "fair-esm==2.0.0"), ("pyhmmer", pyhmmer_ver_str), ("ptflops", "ptflops==0.6.6"),
    ("rna_fm", "rna-fm"), ("openpyxl", "openpyxl"), ("joblib", "joblib==1.0.1"),
    ("pandas", "pandas==1.3.5"), ("matplotlib", "matplotlib==3.5.3"), ("sklearn", "scikit-learn==0.24.0"),
    ("py3Dmol", "py3Dmol"), ("gemmi", "gemmi")
]

# --- Helper Functions ---
def get_package_error(import_name):
    try:
        importlib.invalidate_caches()
        spec = importlib.util.find_spec(import_name)
        if spec is None: return "Package not found."
        __import__(import_name)
        return None
    except Exception as e:
        return f"{type(e).__name__}: {str(e)}"

def is_package_working(import_name):
    if import_name == "rna_fm": return True
    return get_package_error(import_name) is None

def smart_install(import_name, pkg_with_version):
    if import_name == "rna_fm":
        subprocess.run(f"pip install {pkg_with_version} -q", shell=True)
        return None

    if is_package_working(import_name):
        return None

    subprocess.run(f"pip install {pkg_with_version} -q", shell=True, capture_output=True)
    if is_package_working(import_name): return None

    base_pkg = re.split('==|>=|<=|<|>', pkg_with_version)[0]
    res2 = subprocess.run(f"pip install {base_pkg} --upgrade -q", shell=True, capture_output=True, text=True)

    final_err = get_package_error(import_name)
    if final_err:
        combined_output = (res2.stdout + "\n" + res2.stderr).strip()
        return f"Pip Output: {combined_output}\nImport Error: {final_err}"
    return None

def check_weights():
    for f in group_a:
        if not os.path.exists(os.path.join(target_a, f)): return False
    for f in group_b:
        if not os.path.exists(os.path.join(target_b, f)): return False
    return True

def adjust_cryoatom_config(config_path="/content/CryoAtom/CryoAtom2/config.json"):
    if not torch.cuda.is_available():
        return None

    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    if not os.path.exists(config_path):
        print(f"Error: Config file not found at {config_path}")
        return None
    if total_vram_gb > 36:
        print("VRAM > 36GB. Updating config for high speed...")
        with open(config_path, 'r') as f:
            config = json.load(f)
        config['RUNet_args']['batch_size'] = 10
        config['CryoNet_args']['crop_length'] = 600
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)

        # print("Successfully updated: batch_size=10, crop_length=600")
    else:
        print("VRAM <= 36GB. Keeping default config.")
# --- Main Logic ---
install_logs = {}
is_repo_ready = os.path.exists('/content/CryoAtom')
are_deps_ready = all(is_package_working(imp) for imp, _ in deps)
is_core_ready = is_package_working("CryoAtom2")

if is_repo_ready and are_deps_ready and is_core_ready:
    set_progress(100, "Environment and model files already configured.", "#2ecc71")
else:
    # Step 1: Repo
    if not is_repo_ready:
        set_progress(5, "Step 1/5: Cloning repository...")
        subprocess.run("git clone https://github.com/YangLab-SDU/CryoAtom.git -q", shell=True)
        file_path = "/content/CryoAtom/CryoAtom2/utils/hmmer_search.py"
        if os.path.exists(file_path):
          with open(file_path, 'r', encoding='utf-8') as f:
              content = f.read()
              content = content.replace('hit.name.decode("utf-8")', '_to_str(hit.name)')
              content = content.replace('hit.accession.decode("utf-8")', '_to_str(hit.accession)')
              content = content.replace("seq.name.decode('utf-8')", '_to_str(seq.name)')
              content = content.replace('sss2.name.decode("utf-8")', '_to_str(sss2.name)')

              with open(file_path, 'w', encoding='utf-8') as f:
                  f.write(content)
    # Step 1.1: adjust config file
    adjust_cryoatom_config()
    # Step 2: Dependencies
    set_progress(10, "Step 2/5: Checking and installing dependencies...")
    smart_install("tqdm", "tqdm")
    smart_install("gdown", "gdown")
    for i, (imp_name, pkg_name) in enumerate(deps):
        progress = 10 + int((i / len(deps)) * 45)
        set_progress(progress, f"Step 2/5: Processing {pkg_name}...")
        log = smart_install(imp_name, pkg_name)
        if log: install_logs[imp_name] = log



    # Step 3: Core
    if not is_package_working("CryoAtom2"):
        set_progress(55, "Step 3/5: Installing CryoAtom2 core...")
        old_cwd = os.getcwd()
        os.chdir("/content/CryoAtom")
        res_core = subprocess.run("python setup.py install", shell=True, capture_output=True, text=True)
        if res_core.returncode != 0 or not is_package_working("CryoAtom2"):
            install_logs["CryoAtom2"] = res_core.stdout + "\n" + res_core.stderr
        os.chdir(old_cwd)

    # Step 4: Weights
    if not check_weights():
        set_progress(70, "Step 4/5: Downloading Model Weights...")
        os.makedirs(target_a, exist_ok=True)
        os.makedirs(target_b, exist_ok=True)
        import gdown
        gdown.download_folder(id="1BUWOsEXYyX85ClZvNjzeTCaTKQd9UVF9", output=target_a, quiet=True, remaining_ok=True)
        for root_dir, _, files in os.walk(target_a, topdown=False):
            for f in files:
                src_file = os.path.join(root_dir, f)
                if f in group_a and src_file != os.path.join(target_a, f): os.replace(src_file, os.path.join(target_a, f))
                elif f in group_b: os.replace(src_file, os.path.join(target_b, f))
            if root_dir != target_a and not os.listdir(root_dir): os.rmdir(root_dir)

    # Step 5: Final Report
    set_progress(95, "Step 5/5: Final validation...")
    getp_path = "/content/CryoAtom/CryoAtom2/utils/getp"
    if os.path.exists(getp_path): os.chmod(getp_path, 0o755)

    if not install_logs and is_package_working("CryoAtom2"):
        set_progress(100, "Setup Completed Successfully!", "#2ecc71")
    else:
        set_progress(100, f"Setup finished with errors.", "#e74c3c")
        print("\n" + "!"*60)
        print("DETAILED ERROR REPORT (Debug info for missing packages):")
        print("!"*60)
        for pkg, log in install_logs.items():
            print(f"\n[FAILED] Package: {pkg}")
            print("-" * 30)
            print(log)
            print("-" * 30)

In [ ]:
#@title # Step 2: Configure Inference & Data Upload (Rerun to clear the form) { display-mode: "form" }
import os
import base64
import subprocess
import requests
import urllib3
from google.colab import output
from IPython.display import HTML, display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

if 'final_cmd' not in globals():
    final_cmd = ""
command_ready = False

# Example Data Configuration - All dirs set to /content
EXAMPLES = {
    "mode1": {
        "dir": "/content",
        "url": "https://yanglab.qd.sdu.edu.cn/CryoAtom/download/7xht.zip",
        "files": {
            "map": "emd_33198.map",
            "ps": "protein.fasta",
            "rs": "rna.fasta",
            "ds": "dna.fasta"
        }
    },
    "mode2": {
        "dir": "/content",
        "url": "https://yanglab.qd.sdu.edu.cn/CryoAtom/download/9enb.zip",
        "files": {
            "map": "emd_19830.map",
            "pf": "Homo_sapiens_prot.fasta",
            "nf": "Homo_sapiens_rna.fna"
        }
    }
}

def download_data(mode):
    cfg = EXAMPLES[mode]
    target_dir = cfg["dir"] # This is /content

    # Check if the map file already exists in /content
    first_file = os.path.join(target_dir, cfg["files"]["map"])
    if os.path.exists(first_file):
        return

    output.eval_js(f"setUIProgress(0, 'Connecting to server for {mode}...', true)")
    zip_path = os.path.join(target_dir, f"{mode}_data.zip")

    try:
        response = requests.get(cfg["url"], stream=True, verify=False, timeout=60)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))

        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        p = int((downloaded / total_size) * 100)
                        output.eval_js(f"setUIProgress({p}, 'Downloading example data: {p}%', true)")

        output.eval_js("setUIProgress(100, 'Extracting files (flattened)...', true)")
        # -oj: junk paths (do not create folders from zip), extract directly to target_dir
        subprocess.run(f"unzip -oj {zip_path} -d {target_dir}", shell=True, capture_output=True)
        if os.path.exists(zip_path): os.remove(zip_path)
        output.eval_js("setUIProgress(100, 'Example data ready in /content!', true)")

    except Exception as e:
        output.eval_js(f"setUIProgress(0, 'Error: {str(e)[:40]}', true)")

    output.eval_js("setTimeout(() => { document.getElementById('progress_container').style.display = 'none'; }, 2000)")

html_code = f"""
<div id="cryo-portal" style="padding: 25px; background: #ffffff; border: 1px solid #e0e0e0; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05); max-width: 1000px;">
    <h2 style="color: #1a73e8; margin-top: 0; border-bottom: 2px solid #e8f0fe; padding-bottom: 10px;">CryoAtom Configuration Portal</h2>

<!-- Mode Selection Area -->
<div style="margin-bottom: 25px; display: flex; align-items: center; gap: 30px; background: #eef2f7; padding: 20px; border-radius: 10px; border: 1px solid #d1d9e6;">
    <span style="font-weight: bold; color: #1a73e8; font-size: 18px;">Execution Mode:</span>
    <label style="font-size: 18px; cursor: pointer; color: #3c4043; font-weight: 500;">
        <input type="radio" name="work_mode" value="mode1" checked onchange="switchMode('mode1')" style="transform: scale(1.3); margin-right: 8px;">
        Model Building
    </label>
    <label style="font-size: 18px; cursor: pointer; color: #3c4043; font-weight: 500;">
        <input type="radio" name="work_mode" value="mode2" onchange="switchMode('mode2')" style="transform: scale(1.3); margin-right: 8px;">
        Sequence Identification
    </label>
</div>

<div style="margin-bottom: 20px;">
    <label style="font-weight: bold; color: #3c4043;">Job Name (Output Folder):</label><br>
    <input type="text" id="job_name" value="My_CryoAtom_Task" style="width: 100%; padding: 10px; margin-top: 8px; border: 1px solid #dadce0; border-radius: 6px; box-sizing: border-box;">
</div>

<div style="margin-bottom: 25px; padding: 12px; background: #f1f8e9; border-radius: 6px; border: 1px dashed #2e7d32;">
    <label style="cursor: pointer; color: #2e7d32; font-weight: bold; font-size: 15px;">
        <input type="checkbox" id="is_test" onchange="toggleTest(this)"> Use Example Test Data
    </label>
</div>

<!-- Progress Container -->
<div id="progress_container" style="display: none; margin-bottom: 20px; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #1a73e8;">
    <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
        <span id="status_msg" style="font-size: 13px; color: #1a73e8; font-weight: bold;">Ready</span>
        <span id="progress_percent" style="font-size: 13px; color: #666;">0%</span>
    </div>
    <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; overflow: hidden;">
        <div id="progress_bar" style="width: 0%; height: 100%; background-color: #2ecc71; transition: width 0.1s;"></div>
    </div>
</div>

<!-- Mode 1 Inputs (Model Building) -->
<div id="section_mode1">
    <h4 style="color: #1a73e8; margin-bottom: 15px; border-left: 5px solid #1a73e8; padding-left: 10px;">Map & Sequences Input</h4>
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;">
        <div>
            <label style="font-size: 13px; font-weight: 600;">Density Map (.map/.mrc) <span style="color:red">*</span></label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="map_path" placeholder="input map" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_map')" style="background:#1a73e8; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_map" style="display:none" onchange="startChunkedUpload(this, 'map_path')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">Protein Sequence (.fasta)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="prot_path" placeholder="protein.fasta" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_prot')" style="background:#1a73e8; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_prot" style="display:none" onchange="startChunkedUpload(this, 'prot_path')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">RNA Sequence (.fasta)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="rna_path" placeholder="rna.fasta" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_rna')" style="background:#1a73e8; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_rna" style="display:none" onchange="startChunkedUpload(this, 'rna_path')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">DNA Sequence (.fasta)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="dna_path" placeholder="dna.fasta" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_dna')" style="background:#1a73e8; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_dna" style="display:none" onchange="startChunkedUpload(this, 'dna_path')">
        </div>
    </div>
</div>

<!-- Mode 2 Inputs (Sequence Identification) -->
<div id="section_mode2" style="display:none;">
    <h4 style="color: #34a853; margin-bottom: 15px; border-left: 5px solid #34a853; padding-left: 10px;">Identification Configuration</h4>
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;">
        <div>
            <label style="font-size: 13px; font-weight: 600;">Density Map (.map/.mrc) <span style="color:red">*</span></label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="map_path_m2" placeholder="input map" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_map_m2')" style="background:#34a853; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_map_m2" style="display:none" onchange="startChunkedUpload(this, 'map_path_m2')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">Protein Database (.fasta/.fa/.faa)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="pf_path" placeholder="Homo_sapiens_prot.fasta" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_pf')" style="background:#34a853; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_pf" style="display:none" onchange="startChunkedUpload(this, 'pf_path')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">Nucleic Acid Database (.fasta/.fa/.fna)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="nf_path" placeholder="Homo_sapiens_rna.fna" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_nf')" style="background:#34a853; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_nf" style="display:none" onchange="startChunkedUpload(this, 'nf_path')">
        </div>
        <div>
            <label style="font-size: 13px; font-weight: 600;">Backbone Atoms (.cif/.pdb)</label>
            <div style="display: flex; margin-top: 5px;"><input type="text" id="r_path" placeholder="Optional backbone file" style="flex:1; padding:8px; border:1px solid #dadce0; border-radius:4px 0 0 4px;"><button onclick="clickHidden('file_r')" style="background:#34a853; color:white; border:none; padding:0 12px; border-radius:0 4px 4px 0; cursor:pointer;">Upload</button></div>
            <input type="file" id="file_r" style="display:none" onchange="startChunkedUpload(this, 'r_path')">
        </div>
    </div>
</div>

<button onclick="saveParams()" id="confirm_btn" style="width: 100%; padding: 16px; background: #1a73e8; color: #fff; border: none; border-radius: 8px; font-size: 18px; font-weight: bold; cursor: pointer; box-shadow: 0 4px 6px rgba(26,115,232,0.3); transition: background 0.3s; margin-top: 10px;">CONFIRM JOB</button>
</div>

<script>
    let currentMode = 'mode1';

    function switchMode(mode) {{
        currentMode = mode;
        document.getElementById('section_mode1').style.display = mode === 'mode1' ? 'block' : 'none';
        document.getElementById('section_mode2').style.display = mode === 'mode2' ? 'block' : 'none';

        // Reset confirmation UI
        document.getElementById('confirm_btn').style.background = '#1a73e8';
        document.getElementById('confirm_btn').innerText = 'CONFIRM JOB';
        document.getElementById('is_test').checked = false;
        toggleTest(document.getElementById('is_test'));
    }}

    window.setUIProgress = function(percent, message, show) {{
        const container = document.getElementById('progress_container');
        const bar = document.getElementById('progress_bar');
        const msg = document.getElementById('status_msg');
        const pct = document.getElementById('progress_percent');
        if (show) container.style.display = 'block';
        bar.style.width = percent + '%';
        msg.innerText = message;
        pct.innerText = percent + '%';
    }}

    function toggleTest(cb) {{
        const jobInput = document.getElementById('job_name');

        if(cb.checked) {{
            jobInput.disabled = true;
            jobInput.style.backgroundColor = "#f0f0f0";

            if(currentMode === 'mode1') {{
                jobInput.value = "7xht";
                document.getElementById('map_path').value = '/content/{EXAMPLES["mode1"]["files"]["map"]}';
                document.getElementById('prot_path').value = '/content/{EXAMPLES["mode1"]["files"]["ps"]}';
                document.getElementById('rna_path').value = '/content/{EXAMPLES["mode1"]["files"]["rs"]}';
                document.getElementById('dna_path').value = '/content/{EXAMPLES["mode1"]["files"]["ds"]}';
            }} else {{
                jobInput.value = "9enb";
                document.getElementById('map_path_m2').value = '/content/{EXAMPLES["mode2"]["files"]["map"]}';
                document.getElementById('pf_path').value = '/content/{EXAMPLES["mode2"]["files"]["pf"]}';
                document.getElementById('nf_path').value = '/content/{EXAMPLES["mode2"]["files"]["nf"]}';
                document.getElementById('r_path').value = "";
            }}
        }} else {{
            jobInput.disabled = false;
            jobInput.style.backgroundColor = "#fff";
            jobInput.value = "My_CryoAtom_Task";
            // Clear all paths
            ['map_path', 'prot_path', 'rna_path', 'dna_path', 'map_path_m2', 'pf_path', 'nf_path', 'r_path'].forEach(id => {{
                document.getElementById(id).value = "";
            }});
        }}
    }}

    function clickHidden(id) {{ document.getElementById(id).click(); }}

    async function startChunkedUpload(input, targetId) {{
        const file = input.files[0];
        if (!file) return;
        setUIProgress(0, "Streaming: " + file.name, true);
        const chunkSize = 1024 * 1024;
        const totalChunks = Math.ceil(file.size / chunkSize);

        // Pass "content" as jobName to place file in /content/
        await google.colab.kernel.invokeFunction('notebook.InitUpload', [file.name], {{}});
        for (let i = 0; i < totalChunks; i++) {{
            const start = i * chunkSize;
            const end = Math.min(file.size, start + chunkSize);
            const chunk = file.slice(start, end);
            const base64Chunk = await new Promise((resolve) => {{
                const reader = new FileReader();
                reader.onload = () => resolve(reader.result.split(',')[1]);
                reader.readAsDataURL(chunk);
            }});
            await google.colab.kernel.invokeFunction('notebook.WriteChunk', [base64Chunk], {{}});
            setUIProgress(Math.round(((i + 1) / totalChunks) * 100), "Streaming: " + file.name, true);
        }}
        await google.colab.kernel.invokeFunction('notebook.FinalizeUpload', [targetId], {{}});
        setTimeout(() => {{ document.getElementById('progress_container').style.display = 'none'; }}, 2000);
        input.value = "";
    }}

    window.setPathValue = function(inputId, path) {{ document.getElementById(inputId).value = path; }}

    function saveParams() {{
        const params = {{
            mode: currentMode,
            o:    document.getElementById('job_name').value,
            v:    currentMode === 'mode1' ? document.getElementById('map_path').value : document.getElementById('map_path_m2').value,
            ps:   document.getElementById('prot_path').value,
            rs:   document.getElementById('rna_path').value,
            ds:   document.getElementById('dna_path').value,
            pf:   document.getElementById('pf_path').value,
            nf:   document.getElementById('nf_path').value,
            r:    document.getElementById('r_path').value,
            is_test: document.getElementById('is_test').checked
        }};
        google.colab.kernel.invokeFunction('notebook.BuildCommand', [params], {{}});
    }}
</script>
"""

def init_upload(filename):
    global current_file_path
    # Force upload to /content/
    current_file_path = os.path.join("/content", filename)
    with open(current_file_path, "wb") as f: pass

def write_chunk(base64_chunk):
    global current_file_path
    with open(current_file_path, "ab") as f:
        f.write(base64.b64decode(base64_chunk))

def finalize_upload(target_id):
    global current_file_path
    output.eval_js(f"window.setPathValue('{target_id}', '{current_file_path}')")

def build_command(p):
    global final_cmd, command_ready

    if p.get('is_test'):
        download_data(p['mode'])

    if not p['v']:
        output.eval_js("alert('Density Map is required!')")
        return

    # Output directory remains the job name directory
    job_dir = f"/content/{p['o']}"
    os.makedirs(job_dir, exist_ok=True)
    script = "/content/CryoAtom/CryoAtom2/build.py"

    cmd = f"python -W ignore  {script} -v {p['v']} -o {job_dir} -d 0"

    # Argument mapping
    if p['mode'] == 'mode1':
        mapping = {'ps':'-ps', 'rs':'-rs', 'ds':'-ds'}
    else:
        mapping = {'pf':'-pf', 'nf':'-nf', 'r':'-r'}

    for key, flag in mapping.items():
        if p.get(key) and p[key].strip() != "":
            cmd += f" {flag} {p[key]}"

    final_cmd = cmd
    command_ready = True
    output.eval_js("document.getElementById('confirm_btn').style.background = '#f39c12'; document.getElementById('confirm_btn').innerText = 'JOB CONFIGURED';")
    print(f"\n[Configuration Ready]")
    print(f"Input path: {p['v']}")
    print(f"Output path: {job_dir}")
    print(f"Command: {final_cmd}")

output.register_callback('notebook.InitUpload', init_upload)
output.register_callback('notebook.WriteChunk', write_chunk)
output.register_callback('notebook.FinalizeUpload', finalize_upload)
output.register_callback('notebook.BuildCommand', build_command)

display(HTML(html_code))

In [ ]:
#@title # Step 3: Modeling { display-mode: "form" }
import os
import subprocess
import warnings
import torch
warnings.filterwarnings("ignore", category=FutureWarning)

def run_task():
    global final_cmd

    if not final_cmd:
        print("Error: No command found. Please finish Step 1 first!")
        return

    print(f"Execution Started...")
    print(f"CMD: {final_cmd}")
    print("-" * 60)
    %cd /content/CryoAtom
    process = subprocess.Popen(final_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode == 0:
        print(f"\nTask finished successfully.")
    else:
        print(f"\nTask failed with return code {process.returncode}")

run_task()

In [ ]:
#@title #Step 4: 3D Structure Preview & Interactive Download { display-mode: "form" }

Coloring = "Molecule Type" #@param ["Molecule Type", "Confidence Score", "Residue", "Chain"]
Download_Result = True #@param {type:"boolean"}

import py3Dmol, os, re, gemmi, shutil, zipfile, base64
from google.colab import files, output
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

# --- Logic: Find the result files ---
cmd = globals().get('final_cmd', "")
path = None
target_dir = None

if cmd:
    try:
        d = re.search(r'-o\s+([^\s\'"]+)', cmd).group(1).strip("'\"")
        target_dir = d
        path = os.path.join(d, f"{d.split('/')[-1]}.cif")
    except: pass

if not path or not os.path.exists(path):
    cifs = sorted([f for f in os.listdir('.') if f.endswith('.cif')], key=os.path.getmtime)
    path = cifs[-1] if cifs else None
    if path: target_dir = os.path.dirname(path) if os.path.dirname(path) else "."

# --- UI ---
html_ui = """
<div id="download-portal" style="padding: 20px; background: #ffffff; border: 1px solid #e0e0e0; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05); margin-bottom: 20px;">
    <div id="dl_progress_container" style="display: none; margin-bottom: 10px;">
        <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
            <span id="dl_status_msg" style="font-size: 13px; color: #1a73e8; font-weight: bold;">Preparing...</span>
            <span id="dl_progress_percent" style="font-size: 13px; color: #666;">0%</span>
        </div>
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; overflow: hidden;">
            <div id="dl_progress_bar" style="width: 0%; height: 100%; background-color: #2ecc71; transition: width 0.1s;"></div>
        </div>
    </div>
    <button id="dl_btn" onclick="triggerDownload()" style="display:none; width: 100%; padding: 12px; background: #1a73e8; color: #fff; border: none; border-radius: 8px; font-size: 16px; font-weight: bold; cursor: pointer;">DOWNLOAD RESULTS (.ZIP)</button>
</div>

<script>
    window.setDLProgress = function(percent, message, show) {
        const container = document.getElementById('dl_progress_container');
        document.getElementById('dl_progress_bar').style.width = percent + '%';
        document.getElementById('dl_status_msg').innerText = message;
        document.getElementById('dl_progress_percent').innerText = percent + '%';
        container.style.display = show ? 'block' : 'none';
    };

    async function triggerDownload() {
        setDLProgress(0, "Initializing stream...", true);
        google.colab.kernel.invokeFunction('notebook.DownloadZip', [], {});
    }

    let zipChunks = [];
    window.receiveChunk = function(base64Chunk, isLast, fileName) {
        zipChunks.push(Uint8Array.from(atob(base64Chunk), c => c.charCodeAt(0)));
        if (isLast) {
            const blob = new Blob(zipChunks, {type: 'application/zip'});
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.href = url;
            a.download = fileName;
            a.click();
            window.URL.revokeObjectURL(url);
            zipChunks = [];
            setDLProgress(100, "Download Complete!", true);
            setTimeout(() => { document.getElementById('dl_progress_container').style.display = 'none'; }, 3000);
        }
    };
</script>
"""
display(HTML(html_ui))

# --- Download Function ---
def download_zip_callback():
    if not target_dir or not os.path.exists(target_dir): return
    zip_name = os.path.basename(target_dir.rstrip('/')) + ".zip"
    zip_path = os.path.join(target_dir, zip_name)

    if not os.path.exists(zip_path):
        output.eval_js(f"setDLProgress(0, 'Compressing files...', true)")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            files_to_zip = []
            for root, _, fs in os.walk(target_dir):
                for f in fs:
                    if f != zip_name: files_to_zip.append(os.path.join(root, f))
            for i, f in enumerate(files_to_zip):
                zipf.write(f, os.path.relpath(f, target_dir))
                p = int((i+1)/len(files_to_zip)*100)
                output.eval_js(f"setDLProgress({p}, 'Compressing...', true)")

    file_size = os.path.getsize(zip_path)
    chunk_size = 1024 * 1024
    with open(zip_path, "rb") as f:
        sent = 0
        while True:
            chunk = f.read(chunk_size)
            is_last = len(chunk) < chunk_size or sent + len(chunk) >= file_size
            if not chunk: break
            sent += len(chunk)
            p = int((sent / file_size) * 100)
            b64 = base64.b64encode(chunk).decode()
            output.eval_js(f"receiveChunk('{b64}', {str(is_last).lower()}, '{zip_name}')")
            output.eval_js(f"setDLProgress({p}, 'Streaming to browser...', true)")
            if is_last: break

output.register_callback('notebook.DownloadZip', download_zip_callback)

# --- 3D Visualization ---
if path and os.path.exists(path):
    struct = gemmi.read_structure(path)
    tmp_pdb = f"_v_{os.getpid()}.pdb"
    struct.write_pdb(tmp_pdb)

    with open(tmp_pdb) as f:
        data = f.read()

    view = py3Dmol.view(width=900, height=600)
    view.addModel(data, "pdb")

    if Coloring == "Confidence Score":
        display(Markdown("**Confidence Score: Red (Low) → Blue (High)**"))
        view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})

    elif Coloring == "Residue":
        view.setStyle({'cartoon': {'color': 'spectrum'}})

    elif Coloring == "Chain":
        chains = sorted(set(l[21] for l in data.splitlines() if l.startswith("ATOM")))
        colors = ['#4285F4','#EA4335','#FBBC05','#34A853']
        for i, c in enumerate(chains):
            view.setStyle({'chain': c}, {'cartoon': {'color': colors[i % len(colors)]}})

    elif Coloring == "Molecule Type":
        protein_res = {
            'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS',
            'ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
        }
        dna_rna_res = {
            'DA','DT','DG','DC','A','U','G','C'
        }

        protein_sel = {'resn': list(protein_res)}
        dna_sel = {'resn': list(dna_rna_res)}

        view.setStyle({}, {})
        view.setStyle(protein_sel, {'cartoon': {'color': 'green'}})
        view.setStyle(dna_sel, {'cartoon': {'color': 'orange'}})

    view.zoomTo()
    view.show()
    os.remove(tmp_pdb)

# --- Show download button ---
if Download_Result:
    output.eval_js("document.getElementById('dl_btn').style.display = 'block'")

### 📖 Usage Instructions & Mode Descriptions

#### 🚀 Quick Start (Execute steps sequentially: 1 → 2 → 3)
*   **Setup (Step 1):** Run once per session.
*   **Configure (Step 2):** Enter Job Name and upload/path files. Click CONFIRM JOB.
    *(Note: A dedicated folder `/content/[Job Name]` will be created for your results.)*
*   **Run (Step 3):** Start the modeling process.
*   **Visualize (Step 4):** Preview the predicted structure. After Step 3 completes, run the fourth code block to interactively visualize the 3D model.

#### 💡 Tips:
*   **Use GPU:** Before running the pipeline, please check that the runtime type is set to GPU at `Runtime` -> `Change runtime type`. The default GPU for free users is a T4.  To accelerate your workflow with more powerful GPUs such as the A100, consider upgrading to [Colab Pro+](https://colab.research.google.com/signup).
*   **Output Files:** The results will include `[Job Name].cif` and `[Job Name]_raw.cif`, representing the predicted results and the results without input sequence filtering, respectively.
*   **New Tasks:** You only need to run Step 1 the first time you open the notebook. For subsequent runs, simply execute Step 2 and the following steps when submitting new tasks.
*   **Stability:** Google Colab provides a pre-configured environment with the latest Python and CUDA versions, which may not always be compatible with CryoAtom2's dependencies. If you encounter compatibility issues, we recommend installing CryoAtom2 locally by following the instructions on our [GitHub repository](https://github.com/YangLab-SDU/CryoAtom).

#### 📋 Detailed Information for Running Modes

**Mode 1: Model building**
*   **Density map (required):** Input cryo-EM density map
*   **Protein sequence (optional):** Input protein sequences
*   **RNA sequence (optional):** Input RNA sequences
*   **DNA sequence (optional):** Input DNA sequences

**Mode 2: Sequence Identification**
*   **Density map (required):** Cryo-EM density map with resolution better than 5 Å
*   **Protein sequence database (optional):** A sequence database covering all proteins in the density map
*   **Nucleic acid sequence database (optional):** A sequence database covering all nucleic acids in the density map
*   **Backbone fragments (optional):** Backbone fragments for proteins (C<span>α</span> atoms) or nucleic acids (P atoms) used for local identification. If not provided, *de novo* modeling will be performed.